## Day 4 - Part 2: 내 지식을 똑똑하게 사용하는 AI, RAG (Retrieval-Augmented Generation)

### 개요

Day 4의 첫 파트에서 우리는 생성형 AI와 LLM(Large Language Model)의 경이로운 능력을 살펴보았습니다. 

이 거대 언어 모델들은 방대한 텍스트 데이터를 학습하여 인간과 유사한 언어를 구사하고, 요약, 번역, 창작 등 놀라운 작업들을 수행합니다. 

하지만 이 강력한 LLM에게도 몇 가지 명백한 한계가 존재합니다.

* `지식의 단절(Knowledge Cutoff):` LLM의 지식은 특정 시점의 훈련 데이터에 묶여 있습니다. "어제 발표된 최신 AI 모델의 이름은?"과 같은 질문에 답할 수 없습니다. 

* `환각(Hallucination):` 모델이 잘 모르는 내용에 대해 그럴듯한 거짓 정보를 만들어내는 현상이 발생합니다. 
* `개인 정보 접근 불가:` 우리 회사의 내부 문서나 개인적인 이메일 내용에 기반한 답변은 생성할 수 없습니다. 

마치 똑똑하지만 '오픈북 시험'이 허용되지 않는 학생과 같습니다. 

이 문제를 해결하기 위해 등장한 기술이 바로 `RAG(Retrieval-Augmented Generation)`, 즉 `'검색 증강 생성'` 입니다.

RAG는 LLM에게 '오픈북'을 쥐여주는 것과 같습니다.  

사용자의 질문이 들어오면, LLM이 바로 답변을 생성하는 것이 아니라 먼저 관련된 최신 정보나 특정 지식이 담긴 외부 문서(우리의 '오픈북')에서 필요한 내용을 `검색(Retrieve)`합니다. 

그리고 검색된 신뢰할 수 있는 정보를 바탕으로 답변을 `생성(Generate)`하는 것이죠.  

이 접근법을 통해 LLM은 최신성을 유지하고, 사실에 기반한 답변을 하며, 특정 도메인이나 개인화된 데이터에 대한 Q&A도 가능하게 됩니다.

이번 파트에서는 LLM의 한계를 극복하고 그 활용성을 극대화하는 RAG의 세계를 탐험합니다. 

RAG가 왜 필요하며 어떻게 작동하는지 그 핵심 원리를 파헤치고, 실제 최신 뉴스 기사를 우리의 '지식 베이스'로 삼아 Q&A 봇을 만들어 보겠습니다. 

LangChain 프레임워크와 OpenAI의 GPT-4o 모델, 그리고 Supabase 벡터 데이터베이스를 사용하여, 여러분의 손으로 직접 '외부 지식을 활용하는 똑똑한 AI'를 구현하는 전 과정을 경험하게 될 것입니다.

`이번 파트의 학습 목표:`

* 표준 LLM의 한계를 이해하고 RAG의 필요성을 설명할 수 있습니다. 

* RAG의 핵심 파이프라인인 `인덱싱(Indexing), 검색(Retrieval), 생성(Generation)` 의 3단계를 이해합니다. 
* `임베딩(Embedding)` 과 `벡터 데이터베이스(Vector Database)` 의 개념을 이해하고, 의미 기반 검색의 원리를 설명할 수 있습니다.
* `Supabase`를 벡터 데이터베이스로 설정하고 연결하여 데이터를 저장 및 검색할 수 있습니다.
* `LangChain` 프레임워크를 사용하여 데이터 분할, 임베딩, RAG 체인 구축 등 전체 과정을 조율(orchestration)할 수 있습니다. 
* 최신 뉴스 기사를 바탕으로, OpenAI의 `GPT-4o` 모델을 활용하여 사실에 기반한 Q&A를 수행하는 RAG 시스템을 처음부터 끝까지 구현할 수 있습니다.


### 1. RAG는 어떻게 작동하는가?: 3단계 파이프라인

RAG의 작동 방식은 크게 '준비' 단계와 '실행' 단계로 나눌 수 있으며, 세부적으로는 3개의 핵심 과정으로 구성됩니다. 

<img src="https://www.ncloud-forums.com/uploads/monthly_2024_04/877428843_rag.png.fe86ce26c424ea7cd9b347adfe2243f5.png">

`1. 인덱싱 (Indexing): 지식 창고 만들기`

이 단계는 우리의 '오픈북'을 만드는 과정입니다. LLM이 참고할 문서들(PDF, TXT, 웹페이지 등)을 미리 처리하여 검색하기 좋은 형태로 '지식 창고'에 저장합니다. 

* `문서 로드 (Load):` 먼저 지식 베이스가 될 문서들을 불러옵니다.

* `분할 (Split):` 너무 긴 문서는 LLM이 한 번에 처리하기 어렵기 때문에, 의미적으로 연관된 작은 단위(Chunk)로 나눕니다.
* `임베딩 (Embed):` 각 텍스트 조각(Chunk)을 `임베딩 모델`을 사용하여 '의미를 압축한 숫자 벡터'로 변환합니다.  '사과'와 '과일'이라는 단어는 이 벡터 공간에서 서로 가까운 위치에 있게 됩니다.
* `저장 (Store):` 변환된 벡터들을 텍스트 원본과 함께 `벡터 데이터베이스(Vector DB)` 에 저장합니다. 벡터 DB는 이 숫자 벡터들을 효율적으로 저장하고, 특정 벡터와 가장 유사한 벡터들을 빠르게 찾아주는 특수한 데이터베이스입니다. 

`2. 검색 (Retrieval): 관련 정보 찾기`

사용자가 질문을 하면, RAG 시스템은 이 질문에 가장 관련 있는 정보를 지식 창고에서 꺼내옵니다.

* 사용자의 질문 역시 인덱싱 때와 `동일한 임베딩 모델`을 사용하여 숫자 벡터로 변환합니다. 
* 이 질문 벡터를 벡터 DB에 보내, 저장된 문서 벡터들 중 가장 '가까운(유사한)' 벡터들을 찾아달라고 요청합니다. 이것이 바로 `의미 기반 검색(Semantic Search)` 입니다. 
* 벡터 DB는 질문의 의미와 가장 유사한 상위 K개의 문서 조각(Chunk)들을 반환합니다.

`3. 생성 (Generation): 검색된 정보로 답변하기`

이제 LLM이 나설 차례입니다. LLM은 검색된 정보(Context)와 사용자의 원본 질문을 함께 받아 답변을 생성합니다.

* 미리 정의된 `프롬프트 템플릿`에 검색된 문서 조각들과 사용자 질문을 결합하여 LLM에게 전달합니다. 
    > 예시 프롬프트:
    > """
    > 아래의 'Context'를 바탕으로 'Question'에 대해 답변해 주세요. Context에 없는 내용은 답변하지 마세요.
    >
    > Context:
    > {검색된 문서 조각들}
    >
    > Question:
    > {사용자의 원본 질문}
    >
    > Answer:
    > """
* LLM은 이 증강된 프롬프트(Augmented Prompt)를 기반으로, 주어진 정보 내에서 사실에 입각한 답변을 생성합니다. 

이 세 단계를 통해 RAG는 LLM이 최신 정보와 특정 도메인 지식을 활용하여 더 정확하고 신뢰도 높은 답변을 생성하도록 돕습니다.


### 2. RAG 구현의 핵심 도구들

이번 실습에서는 다음과 같은 최신 오픈소스 도구들을 활용하여 RAG 시스템을 구축합니다.

* `OpenAI (GPT-4o & Embeddings):` 세계 최고 수준의 성능을 자랑하는 GPT-4o 모델을 답변 생성기로, `text-embedding-3-small` 모델을 의미 벡터 변환기(임베딩)로 사용합니다.
* `Supabase:` 오픈소스 PostgreSQL 기반의 백엔드 서비스 플랫폼으로, `pgvector` 확장을 통해 강력한 벡터 데이터베이스 기능을 제공합니다.  우리는 Supabase를 우리의 영구적인 '지식 창고'로 활용할 것입니다.
* `LangChain:` LLM 애플리케이션 개발을 위한 필수 프레임워크입니다.  데이터 로딩, 분할, 프롬프트 관리, LLM 및 벡터 DB 연동 등 RAG 파이프라인의 복잡한 흐름을 몇 줄의 코드로 간단하게 조율해 줍니다. 

### 3. 종합 실습: 최신 AI 뉴스기사 Q&A 봇 만들기

이제 이론을 바탕으로, 2025년 AI 기술 동향에 대한 최신 뉴스 기사 3개를 지식 베이스로 삼아 무엇이든 물어보면 답변해주는 Q&A 봇을 직접 만들어 보겠습니다.

#### 3.1. 환경 설정 및 라이브러리 설치

먼저 프로젝트에 필요한 라이브러리들을 설치합니다. `python-dotenv`는 우리의 API 키를 안전하게 관리하기 위해 사용됩니다.

In [1]:
!pip install langchain openai langchain_openai langchain_community supabase python-dotenv beautifulsoup4 

  Using cached PyJWT-2.10.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached pytest_mock-3.14.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached h2-4.2.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached StrEnum-0.4.15-py3-none-any.whl.metadata (5.3 kB)
  Using cached iniconfig-2.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 16.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/893.9 kB ? eta -:--:--
   --------------------------------------- 893.9/893.9 kB 20.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 12.1 MB/s eta 0:00:00
Using cached h2-4.2.0-py3-none-any.whl (60 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.1 which is incompatible.
matplotlib 3.7.5 requires numpy<2,>=1.20, but you have numpy 2.3.1 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.3.1 which is incompatible.
pandas 2.1.4 requires numpy<2,>=1.23.2; python_version == "3.11", but you have numpy 2.3.1 which is incompatible.
pycaret 3.3.2 requires numpy<1.27,>=1.21, but you have numpy 2.3.1 which is incompatible.
scipy 1.11.4 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.3.1 which is incompatible.
sktime 0.26.0 requires numpy<1.27,>=1.21, but you have numpy 2.3.1 which is incompatible.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 2.3.1 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, b


#### 3.2. API 키 및 Supabase 정보 설정

OpenAI API 키와 Supabase 접속 정보를 코드에 직접 노출하는 것은 보안상 위험합니다. `.env` 파일을 생성하고 아래와 같이 키를 저장한 뒤, 파이썬 코드에서 불러와 사용합니다.

`(주의: 아래 코드 셀은 `.env` 파일 생성을 위한 예시이며, 실제 환경에서는 직접 파일을 만들어 내용을 입력해야 합니다.)`

In [ ]:
%writefile .env
# .env 파일 예시 (이 셀은 도커환경 사용자들이 사용하세요. 로컬 사용자 들은 .env 파일을 직접 만들거나 수정하세요.)
# OPENAI_API_KEY=sk-proj-1234567890
# SUPABASE_URL=https://1234567890.supabase.co
# SUPABASE_KEY=1234567890


이제 파이썬 코드에서 이 정보들을 안전하게 로드합니다.

In [2]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# .env 파일에 키가 제대로 설정되었는지 확인 (옵션)
# is_key_available = os.getenv("OPENAI_API_KEY") is not None
# print(f"OpenAI API Key is available: {is_key_available}")

True

#### 3.3. 데이터 준비 (최신 AI 뉴스 기사)

실습을 위해 2025년 AI 기술 트렌드에 관한 가상의 뉴스 기사 세 개를 준비했습니다. 실제 프로젝트에서는 웹 스크레이핑, 파일 로더 등을 사용하여 동적으로 데이터를 가져올 수 있습니다.

In [3]:
# 실습용 샘플 뉴스 데이터
news_articles = [
    {
        "title": "AI, 신약 개발 속도 10배 앞당긴다",
        "content": """
        2025년, 인공지능(AI)이 신약 개발 분야에서 혁신을 주도하고 있다. 
        전통적으로 10년 이상 소요되던 신약 개발 기간이 AI 기반 시뮬레이션과 데이터 분석을 통해 평균 1~2년으로 단축될 전망이다. 
        글로벌 제약사 '뉴로젠'은 최근 AI 플랫폼 '제네시스-1'을 활용해 알츠하이머 치료제 후보 물질을 6개월 만에 발굴했다고 발표했다. 
        이 플랫폼은 수백만 개의 화합물 구조를 가상으로 생성하고, 단백질과의 결합 가능성을 예측하여 유효 물질을 빠르게 선별한다. 
        전문가들은 AI가 개인 맞춤형 치료제 개발을 가속화하고, 희귀병 정복의 새로운 희망이 될 것이라고 평가했다.
        """,
        "source": "AI 타임즈"
    },
    {
        "title": "데이터브릭스, 클라우드 비용 최적화 AI 스타트업 '옵티마이즈' 인수",
        "content": """
        데이터 및 AI 기업 데이터브릭스가 클라우드 비용 관리 AI 스타트업 '옵티마이즈(Optimize.AI)'를 5억 달러에 인수한다고 발표했다. 
        옵티마이즈는 AI를 사용해 기업의 클라우드 사용 패턴을 실시간으로 분석하고, 불필요한 자원을 자동으로 축소하거나 저렴한 플랜으로 전환하여 비용을 최대 40%까지 절감하는 솔루션을 제공한다. 
        데이터브릭스는 이번 인수를 통해 자사의 데이터 인텔리전스 플랫폼에 강력한 비용 최적화 기능을 통합, 
        기업들이 데이터 분석 및 AI 모델 운영 비용을 획기적으로 줄일 수 있도록 지원할 계획이라고 밝혔다. 
        이는 생성형 AI 도입이 확산되면서 급증하는 기업들의 클라우드 비용 부담을 해결하기 위한 전략적 행보다.
        """,
        "source": "테크 위클리"
    },
    {
        "title": "온디바이스 AI, 2025년 스마트폰의 새로운 표준으로",
        "content": """
        클라우드를 거치지 않고 스마트폰, 노트북 등 기기 자체에서 AI 연산을 수행하는 '온디바이스 AI(On-device AI)'가 2025년 기술 업계의 최대 화두로 떠올랐다. 
        온디바이스 AI는 빠른 응답 속도, 강화된 개인정보 보호, 인터넷 연결 없는 AI 기능 사용 등의 장점을 가진다. 
        글로벌 스마트폰 제조사 '테크노바'는 차세대 플래그십 모델 'T-25'에 자체 개발한 신경망처리장치(NPU) '뉴로엔진 X'를 탑재, 
        실시간 통역, AI 사진 편집, 스마트 비서 기능을 오프라인에서도 완벽하게 지원한다고 밝혔다. 
        애플과 구글 또한 차세대 모바일 칩에 더욱 강력한 AI 처리 성능을 예고하고 있어, 온디바이스 AI 시장의 주도권 경쟁은 더욱 치열해질 전망이다.
        """,
        "source": "모바일 월드"
    }
]

In [4]:
# LangChain이 인식할 수 있는 Document 객체로 변환
from langchain.docstore.document import Document

documents = [
    Document(
        page_content=article["content"],
        metadata={"title": article["title"], "source": article["source"]}
    ) for article in news_articles
]

print(f"총 {len(documents)}개의 뉴스를 Document 객체로 변환했습니다.")
print("첫 번째 뉴스 기사 내용:")
print(documents[0].page_content)


총 3개의 뉴스를 Document 객체로 변환했습니다.
첫 번째 뉴스 기사 내용:

        2025년, 인공지능(AI)이 신약 개발 분야에서 혁신을 주도하고 있다. 
        전통적으로 10년 이상 소요되던 신약 개발 기간이 AI 기반 시뮬레이션과 데이터 분석을 통해 평균 1~2년으로 단축될 전망이다. 
        글로벌 제약사 '뉴로젠'은 최근 AI 플랫폼 '제네시스-1'을 활용해 알츠하이머 치료제 후보 물질을 6개월 만에 발굴했다고 발표했다. 
        이 플랫폼은 수백만 개의 화합물 구조를 가상으로 생성하고, 단백질과의 결합 가능성을 예측하여 유효 물질을 빠르게 선별한다. 
        전문가들은 AI가 개인 맞춤형 치료제 개발을 가속화하고, 희귀병 정복의 새로운 희망이 될 것이라고 평가했다.
        


#### 3.4. 인덱싱: 문서를 잘게 쪼개 벡터로 변환 후 저장하기

이제 준비된 뉴스 기사(Document 객체)를 RAG 파이프라인의 첫 단계인 인덱싱 과정에 태워봅시다.

`1. 문서 분할 (Split)`

`RecursiveCharacterTextSplitter`를 사용하여 문서를 의미 있는 작은 조각(chunk)으로 나눕니다.

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # 각 청크의 최대 크기
    chunk_overlap=50,   # 청크 간 중복되는 글자 수
    separators=["\n\n", "\n", " ", ""] # 문단, 줄바꿈 순으로 분할 시도
)

# 문서 분할 실행
split_documents = text_splitter.split_documents(documents)

print(f"원본 문서를 총 {len(split_documents)}개의 청크로 분할했습니다.")
print("\n첫 번째 문서의 첫 번째 청크:")
print(split_documents[0].page_content)

원본 문서를 총 3개의 청크로 분할했습니다.

첫 번째 문서의 첫 번째 청크:
2025년, 인공지능(AI)이 신약 개발 분야에서 혁신을 주도하고 있다. 
        전통적으로 10년 이상 소요되던 신약 개발 기간이 AI 기반 시뮬레이션과 데이터 분석을 통해 평균 1~2년으로 단축될 전망이다. 
        글로벌 제약사 '뉴로젠'은 최근 AI 플랫폼 '제네시스-1'을 활용해 알츠하이머 치료제 후보 물질을 6개월 만에 발굴했다고 발표했다. 
        이 플랫폼은 수백만 개의 화합물 구조를 가상으로 생성하고, 단백질과의 결합 가능성을 예측하여 유효 물질을 빠르게 선별한다. 
        전문가들은 AI가 개인 맞춤형 치료제 개발을 가속화하고, 희귀병 정복의 새로운 희망이 될 것이라고 평가했다.


`2. 임베딩 및 벡터 DB 저장 (Embed & Store)`

분할된 텍스트 청크들을 OpenAI의 임베딩 모델로 벡터화하고, 그 결과를 Supabase 벡터 DB에 저장합니다. LangChain의 `SupabaseVectorStore`를 사용하면 이 과정이 매우 간단해집니다.

> 연결이 안되시는 분은 supabase 콘솔에 접속하셔서 해당 프로젝트가 정지(Pause) 되어 있는지 확인하고 재기동(Restore) 해주세요

In [23]:
# 먼저 기본적인 연결 테스트를 해보세요
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_ANON_KEY")

print(f"Supabase URL: {supabase_url}")
print(f"Key exists: {bool(supabase_key)}")

# 기본 연결 테스트
try:
    supabase = create_client(supabase_url, supabase_key)
    # 간단한 쿼리로 연결 확인
    result = supabase.table('embeddings').select('count').execute()
    print("✅ Supabase 연결 성공!")
except Exception as e:
    print(f"❌ Supabase 연결 실패: {e}")

Supabase URL: https://glspyxqdafeodsnvmfxg.supabase.co
Key exists: True
✅ Supabase 연결 성공!


In [ ]:
# create_supabase_embedding_scheme SQL 실행 embeddings table 생성

데이터가 없을 경우 아래 코드 실행

In [24]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores.supabase import SupabaseVectorStore
from supabase.client import create_client

# Supabase 클라이언트 초기화
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_ANON_KEY")
supabase = create_client(supabase_url, supabase_key)

# OpenAI 임베딩 모델 초기화
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 데이터 삽입과 vectorstore 정의를 같이 해줌
# SupabaseVectorStore 설정
# from_documents() 메소드는 문서 분할, 임베딩, Supabase 저장을 한 번에 처리해줍니다.
vector_store = SupabaseVectorStore.from_documents(
    documents=split_documents,
    embedding=embedding_model,
    client=supabase,
    table_name="embeddings", # Supabase에 생성될 테이블 이름
    query_name="match_embeddings"       # 검색에 사용될 DB 함수 이름
)

print(f"'{vector_store.table_name}' 테이블에 임베딩 저장을 완료했습니다.")

'embeddings' 테이블에 임베딩 저장을 완료했습니다.


데이터가 이미 있을 경우 아래 코드 실행

In [25]:
# 데이터 저장 없이 준비
# Supabase에서 기존 임베딩을 로드하여 vector_store 객체 생성하는 방법
vector_store = SupabaseVectorStore(
    client=supabase,
    embedding=embedding_model,
    table_name="embeddings",
    query_name="match_embeddings"
)

print("✅ Supabase에서 기존 임베딩을 로드하여 vector_store를 생성했습니다.")
print(f"테이블명: {vector_store.table_name}")

✅ Supabase에서 기존 임베딩을 로드하여 vector_store를 생성했습니다.
테이블명: embeddings




#### 3.5. 검색 및 생성: RAG 체인 구축과 실행

지식 창고가 준비되었으니, 이제 사용자의 질문을 받아 답변을 생성하는 RAG 체인을 만들 차례입니다.

`1. 리트리버(Retriever) 생성`

먼저, 질문이 들어왔을 때 Supabase에서 관련 문서를 검색해올 '리트리버'를 정의합니다.

In [26]:
# vector_store에서 전체 문서 조회
all_docs = vector_store.similarity_search("", k=1000)  # 빈 쿼리로 모든 문서 검색

print(f"총 {len(all_docs)}개의 문서가 저장되어 있습니다.")
print("\n=== 저장된 문서 목록 ===")
for i, doc in enumerate(all_docs[:5]):  # 처음 5개만 출력
    print(f"문서 {i+1}: {doc.metadata['source']}")
    print(f"내용 미리보기: {doc.page_content[:100]}...")
    print("-" * 50)

if len(all_docs) > 5:
    print(f"... 그리고 {len(all_docs) - 5}개의 문서가 더 있습니다.")


총 3개의 문서가 저장되어 있습니다.

=== 저장된 문서 목록 ===
문서 1: AI 타임즈
내용 미리보기: 2025년, 인공지능(AI)이 신약 개발 분야에서 혁신을 주도하고 있다. 
        전통적으로 10년 이상 소요되던 신약 개발 기간이 AI 기반 시뮬레이션과 데이터 분석을 통...
--------------------------------------------------
문서 2: 테크 위클리
내용 미리보기: 데이터 및 AI 기업 데이터브릭스가 클라우드 비용 관리 AI 스타트업 '옵티마이즈(Optimize.AI)'를 5억 달러에 인수한다고 발표했다. 
        옵티마이즈는 AI를 ...
--------------------------------------------------
문서 3: 모바일 월드
내용 미리보기: 클라우드를 거치지 않고 스마트폰, 노트북 등 기기 자체에서 AI 연산을 수행하는 '온디바이스 AI(On-device AI)'가 2025년 기술 업계의 최대 화두로 떠올랐다. 
  ...
--------------------------------------------------


In [27]:
# vector_store 객체를 리트리버로 변환
retriever = vector_store.as_retriever(
    search_type="similarity", # 유사도 기반 검색
    search_kwargs={'k': 3}    # 가장 유사한 3개 청크를 반환
)

# 리트리버 테스트
query = "신약 개발에 AI가 어떻게 사용돼?"
retrieved_docs = retriever.invoke(query)

print(f"'{query}'에 대한 검색 결과:")
for i, doc in enumerate(retrieved_docs):
    print(f"--- 문서 {i+1} (출처: {doc.metadata['source']}) ---")
    print(doc.page_content)
    print("-" * 20)

'신약 개발에 AI가 어떻게 사용돼?'에 대한 검색 결과:
--- 문서 1 (출처: AI 타임즈) ---
2025년, 인공지능(AI)이 신약 개발 분야에서 혁신을 주도하고 있다. 
        전통적으로 10년 이상 소요되던 신약 개발 기간이 AI 기반 시뮬레이션과 데이터 분석을 통해 평균 1~2년으로 단축될 전망이다. 
        글로벌 제약사 '뉴로젠'은 최근 AI 플랫폼 '제네시스-1'을 활용해 알츠하이머 치료제 후보 물질을 6개월 만에 발굴했다고 발표했다. 
        이 플랫폼은 수백만 개의 화합물 구조를 가상으로 생성하고, 단백질과의 결합 가능성을 예측하여 유효 물질을 빠르게 선별한다. 
        전문가들은 AI가 개인 맞춤형 치료제 개발을 가속화하고, 희귀병 정복의 새로운 희망이 될 것이라고 평가했다.
--------------------
--- 문서 2 (출처: 모바일 월드) ---
클라우드를 거치지 않고 스마트폰, 노트북 등 기기 자체에서 AI 연산을 수행하는 '온디바이스 AI(On-device AI)'가 2025년 기술 업계의 최대 화두로 떠올랐다. 
        온디바이스 AI는 빠른 응답 속도, 강화된 개인정보 보호, 인터넷 연결 없는 AI 기능 사용 등의 장점을 가진다. 
        글로벌 스마트폰 제조사 '테크노바'는 차세대 플래그십 모델 'T-25'에 자체 개발한 신경망처리장치(NPU) '뉴로엔진 X'를 탑재, 
        실시간 통역, AI 사진 편집, 스마트 비서 기능을 오프라인에서도 완벽하게 지원한다고 밝혔다. 
        애플과 구글 또한 차세대 모바일 칩에 더욱 강력한 AI 처리 성능을 예고하고 있어, 온디바이스 AI 시장의 주도권 경쟁은 더욱 치열해질 전망이다.
--------------------
--- 문서 3 (출처: 테크 위클리) ---
데이터 및 AI 기업 데이터브릭스가 클라우드 비용 관리 AI 스타트업 '옵티마이즈(Optimize.AI)'를 5억 달러에 인수한다고 발표했다. 
     

`2. RAG 체인(Chain) 구축`

LangChain의 `create_stuff_documents_chain`과 `create_retrieval_chain`을 사용하여 프롬프트, LLM, 리트리버를 하나의 체인으로 엮습니다.

In [28]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# LLM 모델 설정 (GPT-4o)
# 4o - 범용적으로 사용 시
# 4.1 - 코딩 시
# mini - 간단한 내용
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

# 프롬프트 템플릿 정의
# []를 제목으로 인식하는 경우가 있음
# 영어 token이 더 적음
prompt = ChatPromptTemplate.from_template("""
당신은 'AI 뉴스 전문 앵커'입니다. 주어진 'Context'를 바탕으로 질문에 대해 친절하고 명확하게 답변해주세요.
반드시 Context에 있는 내용만을 근거로 답변해야 합니다.

[Context]
{context}

[Question]
{input}

[Answer]
""")

# 문서들을 프롬프트에 결합하는 체인 생성
Youtube_chain = create_stuff_documents_chain(llm, prompt)

# 리트리버와 QA 체인을 결합하여 최종 RAG 체인 생성
rag_chain = create_retrieval_chain(retriever, Youtube_chain)

print("RAG 체인 생성이 완료되었습니다.")

RAG 체인 생성이 완료되었습니다.


docker ollama model(로컬 모델) 사용 시
- finetuning 거쳐서 사용

In [ ]:
!pip install langchain-ollama -q

In [ ]:
# 올라마 모델 사용
from langchain_ollama import OllamaLLM

# Ollama LLM 모델 설정 (gemma3:4b 모델 사용)
llm = OllamaLLM(model="gemma3:4b", temperature=0.1)

`3. Q&A 실행`

이제 완성된 RAG 체인에 질문을 던져봅시다!

In [ ]:
# supabase에 저장된 문서를 검색해서 답변

In [29]:
# 질문 1
question1 = "데이터브릭스가 최근에 인수한 회사는 어디고, 그 회사는 무슨 일을 해?"
response1 = rag_chain.invoke({"input": question1})

print(f"❓ 질문: {question1}")
print(f"✅ 답변: {response1['answer']}")

print("-" * 50)

# 질문 2
question2 = "온디바이스 AI의 장점은 뭐야?"
response2 = rag_chain.invoke({"input": question2})

print(f"❓ 질문: {question2}")
print(f"✅ 답변: {response2['answer']}")

print("-" * 50)

# 질문 3 (Context에 없는 내용)
question3 = "삼성전자의 최신 AI 기술은 뭐야?"
response3 = rag_chain.invoke({"input": question3})

print(f"❓ 질문: {question3}")
print(f"✅ 답변: {response3['answer']}")

# RAG 체인의 답변과 함께 검색된 Context도 확인할 수 있습니다.
# print("\n'질문 1'에 사용된 Context:")
# for doc in response1['context']:
#     print(doc.page_content)

❓ 질문: 데이터브릭스가 최근에 인수한 회사는 어디고, 그 회사는 무슨 일을 해?
✅ 답변: 데이터브릭스가 최근에 인수한 회사는 '옵티마이즈(Optimize.AI)'입니다. 이 회사는 AI를 사용하여 기업의 클라우드 사용 패턴을 실시간으로 분석하고, 불필요한 자원을 자동으로 축소하거나 저렴한 플랜으로 전환하여 비용을 최대 40%까지 절감하는 솔루션을 제공합니다.
--------------------------------------------------
❓ 질문: 온디바이스 AI의 장점은 뭐야?
✅ 답변: 온디바이스 AI의 장점은 다음과 같습니다. 첫째, 빠른 응답 속도를 제공합니다. 둘째, 강화된 개인정보 보호를 가능하게 합니다. 셋째, 인터넷 연결 없이도 AI 기능을 사용할 수 있습니다. 이러한 장점들은 기기 자체에서 AI 연산을 수행하기 때문에 가능해집니다.
--------------------------------------------------
❓ 질문: 삼성전자의 최신 AI 기술은 뭐야?
✅ 답변: 주어진 Context에는 삼성전자의 최신 AI 기술에 대한 정보가 포함되어 있지 않습니다. 따라서 삼성전자의 최신 AI 기술에 대해 답변드릴 수 없습니다. Context에 포함된 정보로는 '테크노바'의 차세대 플래그십 모델 'T-25'에 탑재된 신경망처리장치(NPU) '뉴로엔진 X'와 관련된 온디바이스 AI 기술, 그리고 AI가 신약 개발 및 클라우드 비용 관리 분야에서의 혁신을 주도하고 있다는 내용이 있습니다.


이것으로 우리는 최신 뉴스 기사라는 외부 지식을 활용하여 정확하고 사실에 기반한 답변을 생성하는 RAG 시스템을 성공적으로 구축했습니다. 

이제 LLM은 더 이상 과거의 지식에 갇혀있지 않고, 우리가 제공하는 어떤 정보든 학습하고 활용할 수 있는 강력한 파트너가 되었습니다.